In [1]:
# ============================================================
# 06_random_forest_rolling_eval.ipynb

# Random Forest with lag blocks of size 4
# Rolling-origin evaluation, horizons 1..4
#
# Methodology:
#   - Train: 2014-2022, imputed training series
#   - Test: 2023, raw observed truth
#   - Model: one Random Forest per country and disease
#   - Features: last 4 observed/predicted values
#   - Multi-step: recursive
#   - Evaluation: explicit, auditable rolling-origin protocol
# ============================================================

import sys
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# --- make src importable ---
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- src imports ---
from src.config import get_project_paths
from src.io_data import read_ari_ili
from src.calendars import make_calendars_from_df, extract_raw_weekly_series
from src.impute import impute_series_weekly
from src.metrics import mae, rmse, mase_scale_seasonal, mase_from_mae

# --- sklearn ---
from sklearn.ensemble import RandomForestRegressor


# ============================================================
# 0. Settings
# ============================================================

M_SEASON = 52
HORIZONS = (1, 2, 3, 4)
MAX_H = max(HORIZONS)
N_LAGS = 4

OK_ARI = ["BE", "BG", "CZ", "DE", "EE", "LT", "RO", "SI"]
OK_ILI = ["BE", "CZ", "EE", "GR", "IE", "LT", "NL", "PL", "RO", "SI"]

RF_PARAMS = dict(
    n_estimators=300,
    max_depth=None,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

MODEL_NAME = f"RandomForest(lags={N_LAGS})"


# ============================================================
# 1. Load data
# ============================================================

paths = get_project_paths(PROJECT_ROOT)
paths.results.mkdir(parents=True, exist_ok=True)

ari_df, ili_df = read_ari_ili(paths.data)

ari_df["truth_date"] = pd.to_datetime(ari_df["truth_date"])
ili_df["truth_date"] = pd.to_datetime(ili_df["truth_date"])

ari_df = ari_df.sort_values(["location", "truth_date"]).reset_index(drop=True)
ili_df = ili_df.sort_values(["location", "truth_date"]).reset_index(drop=True)

cals_ari = make_calendars_from_df(ari_df)
cals_ili = make_calendars_from_df(ili_df)

print("ARI shape:", ari_df.shape)
print("ILI shape:", ili_df.shape)
print("ARI train weeks:", len(cals_ari.train_weeks))
print("ARI test weeks:", len(cals_ari.test_weeks))
print("ILI train weeks:", len(cals_ili.train_weeks))
print("ILI test weeks:", len(cals_ili.test_weeks))


# ============================================================
# 2. Utilities
# ============================================================

def make_lagged_supervised(series: pd.Series, n_lags: int = 4):
    """
    Convert a univariate time series into supervised format:

        X_t = [y_{t-1}, y_{t-2}, ..., y_{t-n_lags}]
        y_t = y_t

    The input must be the available history. Missing values are dropped before
    constructing lagged blocks.
    """

    s = pd.Series(series).astype(float).dropna().reset_index(drop=True)

    X, y = [], []

    for i in range(n_lags, len(s)):
        lags = s.iloc[i - n_lags:i].values[::-1]
        X.append(lags)
        y.append(s.iloc[i])

    if len(X) == 0:
        return np.empty((0, n_lags)), np.empty((0,))

    return np.asarray(X, dtype=float), np.asarray(y, dtype=float)


def recursive_rf_forecast(
    model: RandomForestRegressor,
    history: pd.Series,
    max_h: int = 4,
    n_lags: int = 4
):
    """
    Recursive multi-step forecast.

    Step 1:
      use last n_lags observed values to predict h=1.

    Steps 2..H:
      append previous prediction to pseudo-history and continue recursively.
    """

    hist = pd.Series(history).astype(float).dropna().tolist()

    if len(hist) < n_lags:
        raise ValueError(f"Need at least {n_lags} non-missing observations to forecast.")

    preds = []
    temp_hist = hist.copy()

    for _ in range(max_h):
        x = np.asarray(temp_hist[-n_lags:][::-1], dtype=float).reshape(1, -1)
        yhat = float(model.predict(x)[0])
        preds.append(yhat)
        temp_hist.append(yhat)

    return np.asarray(preds, dtype=float)


# ============================================================
# 3. Correct rolling-origin evaluation
# ============================================================

def rolling_random_forest(
    history_init: pd.Series,
    y_test: pd.Series,
    disease: str,
    location: str,
    n_lags: int = 4,
    horizons=HORIZONS,
    rf_params=None
):
    """
    Correct rolling-origin evaluation.

    
      - The current 2023 week is revealed only if observed.
      - The model is retrained at each origin using all available history.
      - Forecasts are generated for horizons 1..4.
      - Rows are stored for every target date inside the 2023 test calendar.
      - Missing actuals or missing predictions are NOT removed here.
        They are filtered only in metric computation.
    """

    if rf_params is None:
        rf_params = RF_PARAMS

    history = history_init.astype(float).copy()
    results = []
    max_h = int(max(horizons))

    for origin in y_test.index:

        # Reveal current week's truth if observed
        if pd.notna(y_test.loc[origin]):
            history.loc[origin] = float(y_test.loc[origin])

        hist_non_na = history.dropna()

        if len(hist_non_na) <= n_lags:
            y_hat = np.repeat(np.nan, max_h)
        else:
            X_train, y_train = make_lagged_supervised(hist_non_na, n_lags=n_lags)

            if len(X_train) == 0:
                y_hat = np.repeat(np.nan, max_h)
            else:
                try:
                    model = RandomForestRegressor(**rf_params)
                    model.fit(X_train, y_train)
                    y_hat = recursive_rf_forecast(
                        model=model,
                        history=hist_non_na,
                        max_h=max_h,
                        n_lags=n_lags
                    )
                except Exception:
                    y_hat = np.repeat(np.nan, max_h)

        # Store every target inside the 2023 test calendar
        for h in horizons:
            target = origin + pd.Timedelta(weeks=h)

            if target not in y_test.index:
                continue

            actual = y_test.loc[target]

            try:
                pred = y_hat[h - 1]
            except Exception:
                pred = np.nan

            results.append({
                "origin": origin,
                "target": target,
                "disease": disease,
                "location": location,
                "h": int(h),
                "y": float(actual) if pd.notna(actual) else np.nan,
                "yhat": float(pred) if pd.notna(pred) else np.nan,
                "model": MODEL_NAME
            })

    return pd.DataFrame(results)


def summarize_eval(
    df_eval: pd.DataFrame,
    model_name: str,
    disease: str,
    location: str,
    mase_scale: float
):
    """
    Compute metrics by country and horizon.

    Filtering rule:
      - score only where actual y is observed and prediction yhat is finite.
      - n is number of scored predictions.
    """

    rows = []

    for h in HORIZONS:

        sub = df_eval[df_eval["h"] == h].copy()

        sub = sub[sub["y"].notna()].copy()
        sub = sub[sub["yhat"].notna()].copy()

        if len(sub) == 0:
            continue

        m_mae = mae(sub["y"].values, sub["yhat"].values)
        m_rmse = rmse(sub["y"].values, sub["yhat"].values)
        m_mase = mase_from_mae(m_mae, mase_scale)

        rows.append({
            "disease": disease,
            "location": location,
            "h": int(h),
            "model": model_name,
            "MAE": float(m_mae),
            "RMSE": float(m_rmse),
            "MASE": float(m_mase),
            "n": int(len(sub))
        })

    return pd.DataFrame(rows)


def expected_points_from_truth(y_test: pd.Series, disease: str, location: str):
    """
    Independent audit of available test targets.
    This depends only on raw 2023 truth, not on the model.
    """

    rows = []

    for h in HORIZONS:

        n_expected = 0

        for origin in y_test.index:
            target = origin + pd.Timedelta(weeks=h)

            if target not in y_test.index:
                continue

            if pd.notna(y_test.loc[target]):
                n_expected += 1

        rows.append({
            "disease": disease,
            "location": location,
            "h": int(h),
            "expected_actual_points": int(n_expected)
        })

    return pd.DataFrame(rows)


# ============================================================
# 4. Run RF per disease
# ============================================================

def run_rf_for_disease(
    df: pd.DataFrame,
    disease: str,
    ok_locations: list,
    cals,
    n_lags: int = 4,
    rf_params=None
):
    """
    Run local RF benchmark for each country of one disease.
    Returns:
      - country_h table
      - long prediction table
      - expected points table
      - n audit table
    """

    all_country_h = []
    all_long = []
    all_expected = []

    for loc in ok_locations:

        print(f"\n==================== {disease} {loc} ====================")

        train_imputed = impute_series_weekly(
            df[df["location"] == loc][["truth_date", "value"]],
            calendar=cals.train_weeks,
            trim_to_first_obs=True,
            max_gap_knn=8,
            n_lags=8,
            k_neighbors=5
        )

        y_test = extract_raw_weekly_series(df, loc, cals.test_weeks)

        scale = mase_scale_seasonal(train_imputed, m=M_SEASON)

        expected_df = expected_points_from_truth(
            y_test=y_test,
            disease=disease,
            location=loc
        )
        all_expected.append(expected_df)

        df_eval = rolling_random_forest(
            history_init=train_imputed,
            y_test=y_test,
            disease=disease,
            location=loc,
            n_lags=n_lags,
            horizons=HORIZONS,
            rf_params=rf_params
        )

        all_long.append(df_eval)

        country_h = summarize_eval(
            df_eval=df_eval,
            model_name=f"RandomForest(lags={n_lags})",
            disease=disease,
            location=loc,
            mase_scale=scale
        )

        all_country_h.append(country_h)

    country_h_all = (
        pd.concat(all_country_h, ignore_index=True)
        if all_country_h
        else pd.DataFrame(columns=["disease", "location", "h", "model", "MAE", "RMSE", "MASE", "n"])
    )

    long_all = (
        pd.concat(all_long, ignore_index=True)
        if all_long
        else pd.DataFrame(columns=["origin", "target", "disease", "location", "h", "y", "yhat", "model"])
    )

    expected_all = (
        pd.concat(all_expected, ignore_index=True)
        if all_expected
        else pd.DataFrame(columns=["disease", "location", "h", "expected_actual_points"])
    )

    scored = (
        country_h_all
        .groupby(["disease", "location", "h"], as_index=False)
        .agg(scored_points=("n", "sum"))
    )

    n_audit = expected_all.merge(
        scored,
        on=["disease", "location", "h"],
        how="left"
    )

    n_audit["scored_points"] = n_audit["scored_points"].fillna(0).astype(int)
    n_audit["missing_due_to_model_or_pred_nan"] = (
        n_audit["expected_actual_points"] - n_audit["scored_points"]
    )

    return country_h_all, long_all, expected_all, n_audit


# ============================================================
# 5. Execute ARI and ILI
# ============================================================

rf_ari, rf_ari_long, rf_ari_expected, rf_ari_n_audit = run_rf_for_disease(
    df=ari_df,
    disease="ARI",
    ok_locations=OK_ARI,
    cals=cals_ari,
    n_lags=N_LAGS,
    rf_params=RF_PARAMS
)

print("\nARI RF country_h")
display(rf_ari.head())
print("rf_ari shape:", rf_ari.shape)

print("\nARI RF n audit aggregated")
display(
    rf_ari_n_audit
    .groupby(["disease", "h"], as_index=False)
    .agg(
        expected_actual_points=("expected_actual_points", "sum"),
        scored_points=("scored_points", "sum"),
        missing_due_to_model_or_pred_nan=("missing_due_to_model_or_pred_nan", "sum")
    )
)

rf_ili, rf_ili_long, rf_ili_expected, rf_ili_n_audit = run_rf_for_disease(
    df=ili_df,
    disease="ILI",
    ok_locations=OK_ILI,
    cals=cals_ili,
    n_lags=N_LAGS,
    rf_params=RF_PARAMS
)

print("\nILI RF country_h")
display(rf_ili.head())
print("rf_ili shape:", rf_ili.shape)

print("\nILI RF n audit aggregated")
display(
    rf_ili_n_audit
    .groupby(["disease", "h"], as_index=False)
    .agg(
        expected_actual_points=("expected_actual_points", "sum"),
        scored_points=("scored_points", "sum"),
        missing_due_to_model_or_pred_nan=("missing_due_to_model_or_pred_nan", "sum")
    )
)


# ============================================================
# 6. Coverage-aware summaries
# ============================================================

def coverage_aware_summary(df_res: pd.DataFrame, topk=10):
    if df_res.empty:
        return df_res

    g = (
        df_res
        .groupby(["h", "model"], as_index=False)
        .agg(
            n_countries=("location", "nunique"),
            n_points=("n", "sum"),
            mean_MAE=("MAE", "mean"),
            mean_RMSE=("RMSE", "mean"),
            mean_MASE=("MASE", "mean")
        )
        .sort_values(["h", "mean_MASE", "mean_MAE"])
        .reset_index(drop=True)
    )

    return g.groupby("h", group_keys=False).head(topk)


print("\nARI RF — coverage-aware summary")
rf_ari_summary = coverage_aware_summary(rf_ari, topk=10)
display(rf_ari_summary)

print("\nILI RF — coverage-aware summary")
rf_ili_summary = coverage_aware_summary(rf_ili, topk=10)
display(rf_ili_summary)


# ============================================================
# 7. Save outputs
# ============================================================

out_ari = paths.results / "random_forest_ari_rolling_2023.csv"
out_ili = paths.results / "random_forest_ili_rolling_2023.csv"

out_ari_long = paths.results / "random_forest_ari_rolling_2023_long.csv"
out_ili_long = paths.results / "random_forest_ili_rolling_2023_long.csv"

out_ari_expected = paths.results / "random_forest_ari_expected_points_2023.csv"
out_ili_expected = paths.results / "random_forest_ili_expected_points_2023.csv"

out_ari_audit = paths.results / "random_forest_ari_n_audit_2023.csv"
out_ili_audit = paths.results / "random_forest_ili_n_audit_2023.csv"

out_ari_summary = paths.results / "random_forest_ari_horizon_summary_2023.csv"
out_ili_summary = paths.results / "random_forest_ili_horizon_summary_2023.csv"

rf_ari.to_csv(out_ari, index=False)
rf_ili.to_csv(out_ili, index=False)

rf_ari_long.to_csv(out_ari_long, index=False)
rf_ili_long.to_csv(out_ili_long, index=False)

rf_ari_expected.to_csv(out_ari_expected, index=False)
rf_ili_expected.to_csv(out_ili_expected, index=False)

rf_ari_n_audit.to_csv(out_ari_audit, index=False)
rf_ili_n_audit.to_csv(out_ili_audit, index=False)

rf_ari_summary.to_csv(out_ari_summary, index=False)
rf_ili_summary.to_csv(out_ili_summary, index=False)

print("\nSaved:")
for p in [
    out_ari,
    out_ili,
    out_ari_long,
    out_ili_long,
    out_ari_expected,
    out_ili_expected,
    out_ari_audit,
    out_ili_audit,
    out_ari_summary,
    out_ili_summary
]:
    print("-", p)

ARI shape: (6685, 4)
ILI shape: (10646, 4)
ARI train weeks: 469
ARI test weeks: 53
ILI train weeks: 469
ILI test weeks: 53

==================== ARI BE ====================

==================== ARI BG ====================

==================== ARI CZ ====================

==================== ARI DE ====================

==================== ARI EE ====================

==================== ARI LT ====================

==================== ARI RO ====================

==================== ARI SI ====================

ARI RF country_h


,disease,location,h,model,MAE,RMSE,MASE,n
0,ARI,BE,1,RandomForest(lags=4),151.181652,197.729083,0.509368,52
1,ARI,BE,2,RandomForest(lags=4),195.213977,251.365655,0.657723,51
2,ARI,BE,3,RandomForest(lags=4),213.587073,281.737398,0.719627,50
3,ARI,BE,4,RandomForest(lags=4),235.269467,301.724337,0.792680,49
4,ARI,BG,1,RandomForest(lags=4),101.712825,162.321343,0.546715,51


rf_ari shape: (32, 8)

ARI RF n audit aggregated


,disease,h,expected_actual_points,scored_points,missing_due_to_model_or_pred_nan
0,ARI,1,409,409,0
1,ARI,2,401,401,0
2,ARI,3,393,393,0
3,ARI,4,385,385,0



==================== ILI BE ====================

==================== ILI CZ ====================

==================== ILI EE ====================

==================== ILI GR ====================

==================== ILI IE ====================

==================== ILI LT ====================

==================== ILI NL ====================

==================== ILI PL ====================

==================== ILI RO ====================

==================== ILI SI ====================

ILI RF country_h


...


ARI RF — coverage-aware summary


,h,model,n_countries,n_points,mean_MAE,mean_RMSE,mean_MASE
0,1,RandomForest(lags=4),8,409,122.748632,170.801491,0.602586
1,2,RandomForest(lags=4),8,401,173.615089,235.327310,0.835759
2,3,RandomForest(lags=4),8,393,203.740055,267.907983,0.978911
3,4,RandomForest(lags=4),8,385,227.462497,294.340497,1.095855



ILI RF — coverage-aware summary


,h,model,n_countries,n_points,mean_MAE,mean_RMSE,mean_MASE
0,1,RandomForest(lags=4),10,513,33.713285,56.334672,0.590128
1,2,RandomForest(lags=4),10,503,46.140287,71.675315,0.788064
2,3,RandomForest(lags=4),10,493,57.254948,84.412868,0.979384
3,4,RandomForest(lags=4),10,483,66.785039,94.371408,1.112775



Saved:
- C:\Users\aolas\UNI PYTHON\TFG\results\random_forest_ari_rolling_2023.csv
- C:\Users\aolas\UNI PYTHON\TFG\results\random_forest_ili_rolling_2023.csv
- C:\Users\aolas\UNI PYTHON\TFG\results\random_forest_ari_rolling_2023_long.csv
- C:\Users\aolas\UNI PYTHON\TFG\results\random_forest_ili_rolling_2023_long.csv
- C:\Users\aolas\UNI PYTHON\TFG\results\random_forest_ari_expected_points_2023.csv
- C:\Users\aolas\UNI PYTHON\TFG\results\random_forest_ili_expected_points_2023.csv
- C:\Users\aolas\UNI PYTHON\TFG\results\random_forest_ari_n_audit_2023.csv
- C:\Users\aolas\UNI PYTHON\TFG\results\random_forest_ili_n_audit_2023.csv
- C:\Users\aolas\UNI PYTHON\TFG\results\random_forest_ari_horizon_summary_2023.csv
- C:\Users\aolas\UNI PYTHON\TFG\results\random_forest_ili_horizon_summary_2023.csv
